# 09H · Recover and review the original 09B disk

Run this notebook from the folder containing the original 09H notebook and its helpers, using the same spatial CPU kernel. It reads the prepared native tiles and saved model checkpoints, runs the corrected SpotClean adapter, and recalculates the marker readout using each evaluable cell’s entire footprint. The recovery uses a sibling results folder and creates an aggregate ZIP after each pass.

The default runs **one eligible SpotClean tile** so you can check that the adapter works in your R installation. Set `TILES = "all"` on a subsequent run to process the remaining eligible tiles. To recalculate markers and coverage using existing checkpoints, set `TILES = 0`.

In [ ]:
from pathlib import Path
from IPython.display import FileLink, Image, display
import json
import os
import shutil
import subprocess
import sys
import tempfile
import zipfile

import pandas as pd

NOTEBOOK_DIR = Path.cwd().resolve()
RESULT_ROOT = NOTEBOOK_DIR / "09_pilot_results"
RUN_DIR = None         # Set the full path to a previous 09H_fixed_* results folder if needed
TILES = 1              # 1 for a timed first fit, 0 to read existing fits, "all" to continue
RSCRIPT = None         # Optional path to the working Rscript executable
MINUTES = 60           # Limit for each individual R fit

## Recover saved fits

The driver runs in a fresh Python process so Jupyter's cached modules cannot select an older readout. The candidate folder must contain the original `regions` and `fits` directories. Keep your full local results available. The aggregate ZIP alone contains review tables and diagnostics.

In [ ]:
def original_runs(parent):
    return sorted(p for p in parent.glob("09H_fixed_*") if p.is_dir()
                  and not p.name.endswith("_recovered") and (p / "regions").is_dir()
                  and (p / "fits").is_dir())

if RUN_DIR is None:
    candidates = original_runs(RESULT_ROOT)
    if len(candidates) != 1:
        choices = "\n".join(str(p) for p in candidates) or "(none found)"
        raise ValueError(f"Set RUN_DIR to the original 09H result directory. Found under {RESULT_ROOT}:\n{choices}")
    original = candidates[0].resolve()
else:
    original = Path(RUN_DIR).expanduser().resolve()
if original.name.endswith("_recovered") or not all((original / k).is_dir() for k in ("regions", "fits")):
    raise ValueError(f"Expected the previous run's regions/ and fits/ under {original}.")

driver = NOTEBOOK_DIR / "h09fix" / "recover.py"
if not driver.is_file():
    raise FileNotFoundError(f"Extract the full recovery add-on beside the working 09H notebook. Missing {driver}.")
if TILES != "all" and (type(TILES) is not int or TILES < 0):
    raise ValueError('TILES must be 0, a positive integer, or the string "all".')

recovered = original.with_name(original.name + "_recovered")

def zip_stamps():
    return {p: (p.stat().st_ino, p.stat().st_mtime_ns, p.stat().st_size)
            for p in recovered.glob("*.zip") if p.is_file()}

previous_zips = zip_stamps()
command = [sys.executable, "-I", "-u", str(driver),
           "--run-dir", str(original), "--helpers", str(NOTEBOOK_DIR),
           "--tiles", str(TILES), "--minutes", str(MINUTES)]
if RSCRIPT:
    command.extend(["--rscript", str(Path(RSCRIPT).expanduser().resolve())])
print("Original result directory", original, flush=True)
print("Recovery result directory", recovered, flush=True)
process = subprocess.run(command, cwd=NOTEBOOK_DIR, check=False)
print("Recovery exit code", process.returncode)

current_zips = zip_stamps()
new_zips = [p for p in current_zips if current_zips[p] != previous_zips.get(p)]
archive = max(new_zips, key=lambda p: current_zips[p][1]) if new_zips else None
if archive is None:
    print("This pass produced no new review ZIP. The driver output above contains the diagnostic.")

## How much of the disk has verified fits?

The **reviewed** row describes the available raw data. The **SPARKLE** and **SpotClean** rows give each method's coverage. The **paired** row counts tiles and whole cell footprints with verified fits from both methods. A one-tile pilot can establish that SpotClean runs even when the immune-cell or marker counts in that tile are sparse.

In [ ]:
members = set()
def review_table(filename):
    if archive is None or filename not in members:
        return pd.DataFrame()
    with zipfile.ZipFile(archive) as bundle, bundle.open(filename) as stream:
        try:
            return pd.read_csv(stream)
        except pd.errors.EmptyDataError:
            return pd.DataFrame()

if archive is not None:
    with zipfile.ZipFile(archive) as bundle:
        members = set(bundle.namelist())
        if "status.json" in members:
            status = json.loads(bundle.read("status.json"))
            for k in ("state", "reason", "pause_reason", "r_setup_error", "tiles_attempted", "verified_pairs"):
                if k in status:
                    print(k, status[k])
            if status.get("model_states"):
                display(pd.DataFrame(status["model_states"]).fillna(0).astype(int))
coverage = review_table("comparison/domain_coverage.csv")
if not coverage.empty:
    keep = ["scope", "tiles", "reviewed_owned_disk_bins", "reference_disk_bins",
            "whole_footprints", "reference_prior_cells"]
    display(coverage[keep].set_index("scope"))
elif archive is not None:
    print("Coverage table unavailable in this pass. The return ZIP can be shared for diagnosis.")

## Raw and corrected marker counts

The first plot uses the five biological endpoints in the fixed disk, aggregated across cells with a verified SPARKLE fit. The dotted line represents 100% of the observed counts on those same cells. The paired plot appears when at least one row has a measured retention ratio after both fits. Its denominators can be much smaller. Missing ratios indicate unavailable estimates or zero observed UMIs.

Compare myeloid retention in nearby T-supported cells with myeloid retention in macrophage/monocyte labels. The counts below each plot show how much evidence supports each percentage. A drop in Cd4 or CD8 counts alone cannot identify the lineage of an individual cell.

In [ ]:
markers = review_table("comparison/method_coverage.csv")
endpoints = {
    ("near_panT", "myeloid"): "Myeloid in nearby T-supported cells",
    ("macrophage", "myeloid"): "Myeloid in macrophage/monocyte labels",
    ("panT_supported", "panT"): "Pan-T in T-supported cells",
    ("T_label", "Cd4"): "Cd4 in prior T labels",
    ("T_label", "CD8"): "CD8 in prior T labels",
}

def show_figure(filename):
    if filename in members:
        with zipfile.ZipFile(archive) as bundle:
            display(Image(data=bundle.read(filename), width=820))
        return True
    return False

def show_scope(scope, methods):
    part = markers.loc[markers.scope.eq(scope)].copy()
    if part.empty:
        return
    order = {key: i for i, key in enumerate(endpoints)}
    keys = list(zip(part.group, part.panel))
    part["order"] = [order.get(k, len(order)) for k in keys]
    part["endpoint"] = [endpoints.get(k, " / ".join(k)) for k in keys]
    part.sort_values(["order", "genes"], inplace=True)
    columns = ["endpoint", "tiles_with_measured_cells", "cells", "raw_UMIs"]
    if part.endpoint.duplicated().any():
        columns.insert(1, "genes")
    for method in methods:
        key = method.lower()
        percent = method + " retained (%)"
        part[percent] = 100 * pd.to_numeric(part[key + "_retention"], errors="coerce")
        columns.extend([key + "_counts", percent])
    readable = {"tiles_with_measured_cells": "tiles", "raw_UMIs": "observed UMIs",
                "sparkle_counts": "SPARKLE counts", "spotclean_counts": "SpotClean counts"}
    title = {"SPARKLE": "SPARKLE across its verified footprints", "paired": "Both methods on identical footprints"}[scope]
    print(title)
    if not show_figure(f"comparison/{scope.lower()}_marker_retention.png"):
        print("No plotted retention estimates. Inspect the raw counts and availability below.")
    display(part[columns].rename(columns=readable).set_index("endpoint").round(1))

if not markers.empty:
    show_scope("SPARKLE", ["SPARKLE"])
    show_scope("paired", ["SPARKLE", "SpotClean"])
if show_figure("comparison/raw_coreceptor_masks.png"):
    print("Raw counts inside masks include all segmented cell types.")

## Return this pass

The link below includes the driver's aggregate results and the displayed marker plots. It is available for partial and diagnostic runs as well as full runs. Native matrices and per-cell files stay in the study results directory.

In [ ]:
if archive is not None:
    target = NOTEBOOK_DIR / archive.name
    handle, tmp_name = tempfile.mkstemp(prefix=".09h_review_", dir=NOTEBOOK_DIR)
    os.close(handle)
    temp = Path(tmp_name)
    try:
        shutil.copyfile(archive, temp)
        temp.replace(target)
    finally:
        temp.unlink(missing_ok=True)
    print("Return ZIP", target)
    display(FileLink(target.name))